# BanglaKnight starter notebook

BLP 2027 Shared Task 1. This notebook:

1. loads the labeled **train** and **dev** sets,
2. trains a simple baseline (TF-IDF character n-grams + linear SVM),
3. reports the official **Safety Score** on dev,
4. predicts the blind file of the current Codabench phase and saves `my_predictions.csv`.

Then open the **prepare_submission_file** notebook to turn `my_predictions.csv` into the zip you upload.

Labels: 0 Safe, 1 Offensive, 2 Harmful, 3 Severe. Every prompt appears in three forms (Bangla script, romanized, code-mixed) with the same label.

## 1. Setup

Runs on a laptop CPU or in Google Colab in a few minutes. The train and dev files are downloaded from the task website if they are not in the working folder.

**Blind file:** the unlabeled file of the current phase is also downloaded from the task website: `banglaknight_devtest.csv` in the Development phase, `banglaknight_test.csv` in the Test phase (online from Jan 8, 2027).

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report

BLIND_FILE = "banglaknight_devtest.csv"   # Test phase: "banglaknight_test.csv"
SITE = "https://banglaknight.github.io/data/files"

for name in ["banglaknight_train.csv", "banglaknight_dev.csv", BLIND_FILE]:
    if not os.path.exists(name):
        print("downloading", name)
        urllib.request.urlretrieve(f"{SITE}/{name}", name)

train = pd.read_csv("banglaknight_train.csv")
dev = pd.read_csv("banglaknight_dev.csv")
print("train", train.shape, "| dev", dev.shape)
print(train.groupby(["form", "label"]).size().unstack())

## 2. The official metric

The same computation as the Codabench scoring program.

In [ ]:
FORMS = ["bangla", "romanized", "codemixed"]

def safety_score(df, pred):
    """Official metric: mean macro-F1 over the three forms x (1 - USR).
    USR = share of unsafe rows (label > 0) predicted Safe, all forms pooled."""
    y, f, pred = df["label"].values, df["form"].values, np.asarray(pred)
    r = {}
    for fm in FORMS:
        m = f == fm
        r[f"F1 {fm}"] = f1_score(y[m], pred[m], labels=[0, 1, 2, 3], average="macro", zero_division=0)
    r["Average F1"] = float(np.mean([r[f"F1 {fm}"] for fm in FORMS]))
    r["USR"] = float((pred[y > 0] == 0).mean())
    r["Safety Score"] = r["Average F1"] * (1 - r["USR"])
    return r

## 3. Baseline: TF-IDF + linear SVM

Character n-grams (2 to 5, inside word boundaries) work for all three forms without a tokenizer. One model is trained on all forms together.

In [ ]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, max_features=300_000, sublinear_tf=True)
X_train = vec.fit_transform(train["text"])
X_dev = vec.transform(dev["text"])
clf = LinearSVC(C=0.5, class_weight="balanced")
clf.fit(X_train, train["label"])
print("features:", X_train.shape[1])

## 4. Safety Score on dev

In [ ]:
dev_pred = clf.predict(X_dev)
scores = safety_score(dev, dev_pred)
print(pd.Series(scores).round(4).to_string())
print()
print(classification_report(dev["label"], dev_pred, target_names=["Safe", "Offensive", "Harmful", "Severe"], digits=3))

### Ideas to improve

- **Consistency across forms:** the three rows of a prompt share a `prompt_id` and always share the label. Combining the three predictions (for example, taking the most unsafe one) can lower the USR.
- **Fewer missed unsafe prompts:** the USR term punishes unsafe prompts labeled Safe. Shifting the decision threshold of the Safe class, tuned on dev, trades a little precision for a lower USR.
- **Pretrained models:** multilingual or Bangla encoders, and LLM-based classifiers.

## 5. Predict the blind file

In [ ]:
blind = pd.read_csv(BLIND_FILE)
print(BLIND_FILE, blind.shape)
blind_pred = clf.predict(vec.transform(blind["text"]))
pd.DataFrame({"id": blind["id"], "label": blind_pred}).to_csv("my_predictions.csv", index=False)
print("wrote my_predictions.csv; next: the prepare_submission_file notebook")